## Практична робота 4. Чищення набору даних

Варіант 6 — чернігівський інтернет-магазин  
**Прізвище та ім'я:** Войтович Богдан  
**Група:** IT-32


In [27]:
import pandas as pd
import numpy as np

base_price = 690
base_qty = 4
city_variants = ["Чернігів", " Ч е р н і г і в ", "CHERNIHIV"]  # 3 варіанти назви

data = [
    {"місто": city_variants[0], "ціна": base_price, "кількість": base_qty},
    {"місто": city_variants[1], "ціна": f"{base_price} грн", "кількість": np.nan},       # NaN в кількості + рядок у ціні
    {"місто": city_variants[0], "ціна": base_price * 9, "кількість": base_qty},         # викид — ціна в 9 разів більша
    {"місто": city_variants[2], "ціна": f"{base_price} грн", "кількість": base_qty},    # рядок у ціні, інша назва міста
    {"місто": city_variants[0], "ціна": base_price, "кількість": base_qty},
    {"місто": city_variants[0], "ціна": base_price, "кількість": base_qty},
    {"місто": city_variants[0], "ціна": base_price, "кількість": np.nan},               # пропуск в кількості
    {"місто": city_variants[0], "ціна": base_price, "кількість": base_qty},             # дублікати нижче
    {"місто": city_variants[0], "ціна": base_price, "кількість": base_qty},             # дублікат повний рядок з попереднім
]

df = pd.DataFrame(data)
print("Початковий 'брудний' набір:")
print(df)


Початковий 'брудний' набір:
               місто     ціна  кількість
0           Чернігів      690        4.0
1   Ч е р н і г і в   690 грн        NaN
2           Чернігів     6210        4.0
3          CHERNIHIV  690 грн        4.0
4           Чернігів      690        4.0
5           Чернігів      690        4.0
6           Чернігів      690        NaN
7           Чернігів      690        4.0
8           Чернігів      690        4.0


In [28]:
print("Пропуски в кожному стовпці:")
print(df.isna().sum())

# Заповнюємо пропуски кількості медіаною (доречніше за середнє для маленьких наборів)
median_qty = df["кількість"].median()
df["кількість"].fillna(median_qty, inplace=True)

print("\nПісля заповнення пропусків медіаною:")
print(df)


Пропуски в кожному стовпці:
місто        0
ціна         0
кількість    2
dtype: int64

Після заповнення пропусків медіаною:
               місто     ціна  кількість
0           Чернігів      690        4.0
1   Ч е р н і г і в   690 грн        4.0
2           Чернігів     6210        4.0
3          CHERNIHIV  690 грн        4.0
4           Чернігів      690        4.0
5           Чернігів      690        4.0
6           Чернігів      690        4.0
7           Чернігів      690        4.0
8           Чернігів      690        4.0


/tmp/ipykernel_4877/2567436193.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["кількість"].fillna(median_qty, inplace=True)


In [29]:
# Пошук повних дублікатів
print("Кількість повних дублікатів:", df.duplicated().sum())

# Видалення повних дублікатів
df.drop_duplicates(inplace=True)

print("Після видалення дублікатів — рядків залишилось:", len(df))


Кількість повних дублікатів: 5
Після видалення дублікатів — рядків залишилось: 4


In [30]:
# Приведення "ціна" до float (прибираємо " грн")
df["ціна"] = df["ціна"].astype(str).str.replace(" грн", "").astype(float)

# Узгоджуємо назву міста: прибираємо пробіли, переводимо в нижній регістр
df["місто"] = df["місто"].str.replace(" ", "").str.lower()

print("\nПісля приведення типів і узгодження категорій:")
print(df)
print("Унікальні міста після узгодження:", df["місто"].unique())



Після приведення типів і узгодження категорій:
       місто    ціна  кількість
0   чернігів   690.0        4.0
1   чернігів   690.0        4.0
2   чернігів  6210.0        4.0
3  chernihiv   690.0        4.0
Унікальні міста після узгодження: ['чернігів' 'chernihiv']


In [31]:
q1, q3 = df["ціна"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"IQR межі: нижня = {lower_bound}, верхня = {upper_bound}")

outliers = df[(df["ціна"] < lower_bound) | (df["ціна"] > upper_bound)]
print("Викиди за ціною:")
print(outliers)

# Пояснення:
explanation = """
Велике значення ціни в 9 разів більше за базову — це, ймовірно, помилка вводу (зайвий нуль).
Краще сприймати це як помилку, а не реальне замовлення, щоб уникнути викривлення статистики.
"""

print(explanation)


IQR межі: нижня = -1380.0, верхня = 4140.0
Викиди за ціною:
      місто    ціна  кількість
2  чернігів  6210.0        4.0

Велике значення ціни в 9 разів більше за базову — це, ймовірно, помилка вводу (зайвий нуль).
Краще сприймати це як помилку, а не реальне замовлення, щоб уникнути викривлення статистики.



### Відповіді на питання:

1. Заповнення пропусків середнім може сильно спотворити стандартне відхилення, особливо в маленьких наборах, бо чутливе до крайніх значень. Медіана більш устойчива до викидів, тому її застосовують тут.

2. `duplicated()` без subset шукає повні дублікати рядків, з subset — унікальність за вибраними колонками. Результат відрізняється, бо унікальність може враховувати лише ключові поля.

3. Автоматичне видалення поза межами IQR не завжди правильне, бо може вилучити реальні спостереження (наприклад, великі замовлення). Краще аналізувати кожен випадок вручну.
